## **Extracción de Características Textuales (Textual Embeddings)**

En este notebook abordamos la fase de extracción de características para la modalidad de texto (transcripciones de diálogos). Mientras que el audio captura el "cómo" se dice y el vídeo el "gesto" con el que se dice, el texto aporta la semántica y el contexto lingüístico, imprescindibles en nuestra tarea de detección de **Estrés**.

Al igual que en las modalidades anteriores, no utilizaremos el texto en crudo, sino que emplearemos modelos de lenguaje basados en Transformers para generar embeddings contextuales. A diferencia de técnicas tradicionales como Word2Vec, estos modelos utilizan mecanismos de *self-attention* que permiten que la representación de cada palabra dependa de todas las demás palabras de la frase, capturando matices de estrés y frustración.

Dado que todas las transcripciones de nuestro dataset se encuentran en inglés, se han seleccionado los siguientes modelos preentrenados específicamente en este idioma para realizar un estudio comparativo:

1. **BERT** : Es el modelo clásico pero efectivo que revolucionó el NLP al procesar el texto de forma bidireccional.

2. **RoBERTa**: Versión mejorada de **BERT**, que optimiza el proceso de entrenamiento y el tamaño de los datos, demostrando mayor eficacia en la captura de matices emocionales en diálogos conversacionales.

3. **DeBERTa-v3**: Representa el estado del arte actual. Su mecanismo de atención permite separar la información del contenido de la posición de la palabra, lo que permite una mayor capacidad para entender estructuras gramaticales complejas bajo tensión.


**NOTA**: Para cada frase, extraeremos los embeddings de la última capa oculta. El resultado será un conjunto de archivos *.npy* con dimensiones $(T, 768)$, donde $T$ es el número de tokens y 768 es la densidad del vector de características (en los 3 modelos).
Al cargar la versión **Base** de los tres modelos, se obtienen esas 768 dimensiones, alineadas con los modelos más robustos en audio (**Wav2Vec2.0**) y visión (**ViT**), que convergen de forma nativa a **768 dimensiones** en sus versiones base cargadas. Esto facilita la creación de espacio latente común, donde las dimensiones de entrada están equilibradas.

In [2]:
!pip install -q transformers sentencepiece torch tqdm

In [12]:
# Carga previa de todas las librerías y paquetes necesarios
import os
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import random
import sys

# Configuración de GPU/CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

Dispositivo: cuda


In [13]:
BASE_DIR = os.path.expanduser('/workspace') #Directorio base
CSV_PATH = os.path.join(BASE_DIR, 'Multimodal_Stress_Dataset.csv') # CSV global unificado
LOCAL_DATA_ROOT = os.path.join(BASE_DIR, 'EMBEDDINGS_TEXTO') # Directorio donde se almacenarán los features textuales
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

# Carga del dataset global
if os.path.exists(CSV_PATH):
    df_global = pd.read_csv(CSV_PATH)
    display(df_global.head()) 
else:
    print(f"No se encuentra el archivo en {CSV_PATH}")

,Utterance_ID,Dialogue_ID,video_path,audio_path,Transcription,duration,split,target_stress,dataset_origin
0,train_dia0_utt0,0,train_splits/dia0_utt0.mp4,MELD_Audio/train_dia0_utt0.wav,also I was the point person on my company's tr...,5.672333,train,0,MELD
1,train_dia0_utt1,0,train_splits/dia0_utt1.mp4,MELD_Audio/train_dia0_utt1.wav,You must've had your hands full.,1.501500,train,0,MELD
2,train_dia0_utt2,0,train_splits/dia0_utt2.mp4,MELD_Audio/train_dia0_utt2.wav,That I did. That I did.,2.919583,train,0,MELD
3,train_dia0_utt3,0,train_splits/dia0_utt3.mp4,MELD_Audio/train_dia0_utt3.wav,So let's talk a little bit about your duties.,2.752750,train,0,MELD
4,train_dia0_utt4,0,train_splits/dia0_utt4.mp4,MELD_Audio/train_dia0_utt4.wav,My duties? All right.,6.464792,train,0,MELD


----

#### **ESTRUCTURA NOTEBOOK Y EJECUCIÓN**:

Aunque la carga computacional sea menor a los dos notebooks de extracción anteriores, por simetría con estos y para guardar los pesos obtenidos en el servidor para los siguientes entrenamientos, este notebook también se ejecuta en el Servidor NVIDIA DGX al cual se ha tenido acceso para realizar este TFG.

A continuación, se presenta el análisis inicial llevado a cabo, previo a la extracción de características textuales para decidir el tamaño de la ventana de contexto, y se explica y ejecuta el código del bucle final lanzado en el servidor para la extracción de características, cargando en este mismo los distintos modelos mencionados.

Finalmente, se cargan los *embeddings* resultantes para realizar una auditoría final de validación (**sanity check**).

-----

### **Análisis Previo**

### Estrategia de Extracción Multi-Ventana:

Se recuperan los resultados obtenidos en el EDA por cada dataset sobre la distribución del número de palabras por frase/utterance:

* **MELD**:
    - *** ESTADÍSTICAS DE LONGITUD (TOKENIZACIÓN)***
        - Longitud Mínima: 1 palabras
        - Longitud Máxima: 69 palabras
        - Media: 8.01 palabras
        - Percentil 95%: 20 palabras
        - Percentil 99%: 27 palabras

* **IEMOCAP**:
    - *** ESTADÍSTICAS DE LONGITUD (TOKENIZACIÓN)***
        - Longitud Mínima: 1 palabras
        - Longitud Máxima: 100 palabras
        - Media: 11.90 palabras
        - Percentil 95%: 33 palabras
        - Percentil 99%: 47 palabras

Basándonos en estos resultados, se determina que el **99% de las transcripciones** en los datasets MELD e IEMOCAP poseen una longitud de **27** y **47** tokens respectivamente, se ha diseñado una estrategia de extracción para cumplir con los requisitos del estudio de ablación metodológico:

1.  **Ventana Corta ($T=32$ tokens):** Con 32 tokens como ventana de contexto capturamos más del 99% de las transcripciones en MELD de forma completa (sin truncar) y el 95% en IEMOCAP. Con esta ventana corta, capturamos respuestas inmediatas y palabras clave. El objetivo es evaluar si el modelo es capaz de detectar el estrés basándose únicamente en el inicio de la intervención o en frases cortas.
2.  **Ventana Media ($T=64$ tokens):** Con 64 tokens como ventana de contexto, ahora se cubre casi el 100% de las transcripciones en MELD sin truncamiento y más del 99% en IEMOCAP. Es un balance estándar, evitando la introducción excesiva de *padding* que sí ocurriría con ventanas mayores (por ejemplo, de 128 o 256) que sería excesivo en nuestro caso.

**NOTA: A diferencia de las otras dos modalidades, en texto la ventana de contexto sí afecta semánticamente a las características extraídas. En vídeo, se muestrean directamente los 32 frames fijados; en audio, se extrae la secuencia completa y es `dataset.py` quien aplica el truncado/padding en función de la ventana elegida, sin alterar el contenido semántico del embedding. En texto, en cambio, truncar a 32 o a 64 tokens produce representaciones intrínsecamente distintas, ya que el modelo solo "ve" una porción diferente de la transcripción. Por este motivo, la extracción se realiza dos veces, generando dos conjuntos de embeddings independientes (ventana corta y ventana media) para cada modelo textual.**

Por tanto, para garantizar la validez de la comparación entre ambos tamaños de ventana, se realizan **inferencias independientes** (`max_length=32`, `max_length=64`) para cada configuración. Esto evita el "ruido contextual" que se introduciría al recortar artificialmente un tensor generado con una ventana mayor, ya que los mecanismos de *self-attention* de los Transformers (BERT, RoBERTa, DeBERTa) generan dependencias bidireccionales entre todos los tokens procesados.

----

La estructura de directorios, tras la ejecución de este notebook, quedará de la siguiente forma:
* **`EMBEDDINGS_TEXTO`**:

    * **`BERT`**: 

        * **`MELD_32`**: Embeddings de tamaño $(32,768)$ obtenidos de **BERT** para el dataset de **MELD**.

        * **`MELD_64`**: Embeddings de tamaño $(64,768)$ obtenidos de **BERT** para el dataset de **MELD**.

        * **`IEMOCAP_32`**: Embeddings de tamaño $(32,768)$ obtenidos de **BERT** para el dataset de **IEMOCAP**.

        * **`IEMOCAP_64`**: Embeddings de tamaño $(64,768)$ obtenidos de **BERT** para el dataset de **IEMOCAP**.

        * **`EMBEDDINGS_TEXT_BERT_64`**: Aquí se almacenarán los embeddings de tamaño $(64,768)$ obtenidos de **BERT**.


    * **`ROBERTA`**: 

        * **`MELD_32`**: Embeddings de tamaño $(32,768)$ obtenidos de **RoBERTa** para el dataset de **MELD**.

        * **`MELD_64`**: Embeddings de tamaño $(64,768)$ obtenidos de **RoBERTa** para el dataset de **MELD**.

        * **`IEMOCAP_32`**: Embeddings de tamaño $(32,768)$ obtenidos de **RoBERTa** para el dataset de **IEMOCAP**.

        * **`IEMOCAP_64`**: Embeddings de tamaño $(64,768)$ obtenidos de **RoBERTa** para el dataset de **IEMOCAP**.

    
    * **`DEBERTA`**: 

        * **`MELD_32`**: Embeddings de tamaño $(32,768)$ obtenidos de **DeBERTa** para el dataset de **MELD**.

        * **`MELD_64`**: Embeddings de tamaño $(64,768)$ obtenidos de **DeBERTa** para el dataset de **MELD**.

        * **`IEMOCAP_32`**: Embeddings de tamaño $(32,768)$ obtenidos de **DeBERTa** para el dataset de **IEMOCAP**.

        * **`IEMOCAP_64`**: Embeddings de tamaño $(64,768)$ obtenidos de **DeBERTa** para el dataset de **IEMOCAP**.

----
## ***Feed-forward.* Fase de Extracción**.

In [14]:
def extract_features(df_subset, model_name, tokenizer, model, max_len, output_dir):
    """
    Función para extraer características textuales utilizando un modelo de Hugging Face dado.
    
    Args:
        df_subset (pd.DataFrame): DataFrame con las transcripciones y metadatos filtrado por dataset seleccionado
        model_name (str): Nombre del modelo (e.g., 'bert', 'roberta', 'deberta').
        tokenizer: Tokenizador del modelo seleccionado.
        model: Modelo ya cargado y en modo evaluación (inferencia)
        max_len (int): Longitud máxima de la ventana de texto (tokens)
        output_dir (str): Directorio donde se guardarán los embeddings extraídos.
    
    Devuelve:
        None (los embeddings se guardan como archivos .npy en el directorio especificado).
    """
    os.makedirs(output_dir, exist_ok=True)  # Crear el directorio de salida si no existe  

    origen = df_subset['dataset_origin'].iloc[0]

    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Extrayendo características con {model_name} (max_len={max_len}) -- {origen.upper()}", file = sys.stdout, position=0, leave=True):
        file_name = f"{row['dataset_origin']}_{row['Utterance_ID']}.npy".replace("/", "_")

        file_path = os.path.join(output_dir, file_name)

        if os.path.exists(file_path): # Reanudación si se corta la ejecución
            continue

        transcription = row['Transcription']

        # Tokenización y creación de tensores
        inputs = tokenizer(transcription, # Tokenizamos la transcripción
                            return_tensors='pt', # Devolvemos tensores PyTorch
                            truncation=True,  # Truncamos el texto si excede max_len
                            padding='max_length',  # Rellenamos con ceros hasta max_len
                            max_length=max_len) # Establecemos la longitud máxima de la ventana
        inputs = {key: val.to(device) for key, val in inputs.items()}

        # Extracción de características sin calcular gradientes (inferencia):
        with torch.no_grad(): 
            outputs = model(**inputs) # Obtenemos las salidas del modelo (embeddings)
            # Guardamos el embedding de la secuencia completa:
            embedding = outputs.last_hidden_state.squeeze(0).cpu().numpy() 

        # Guardar el embedding como un archivo .npy
        np.save(file_path, embedding)  # Guardamos el embedding en un archivo .npy

In [15]:
# Bucle principal:

# Configuración diccionario con los modelos a utilizar y sus respectivas rutas en Hugging Face:
MODELS_CONFIG = {
    "bert": "bert-base-uncased", # Modelo preentrenado básico con corpus de datos en inglés - 110M parámetros
    "roberta": "roberta-base", # Modelo preentrenado básico con corpus de datos en inglés, con misma arquitectura que BERT (elimina para el preentrenamiento NSP) - 125M parámetros
    "deberta": "microsoft/deberta-v3-base" # EMPLEAMOS LA v3 y no la v1 ya que es la estándar en la literatura
}

# Longitudes de ventana fijadas:
WINDOW_SIZES = [32, 64]

origenes = ['MELD', 'IEMOCAP']

for nombre_modelo, hf_path in MODELS_CONFIG.items(): # --> BERT, ROBERTA, DEBERTA
    # Cargamos el modelo UNA vez solo:
    tokenizador = AutoTokenizer.from_pretrained(hf_path)
    modelo = AutoModel.from_pretrained(hf_path).to(device)
    modelo.eval()
    for max_len in WINDOW_SIZES: # ---> 32, 64
        for origen in origenes: # ----> MELD, IEMOCAP
            df_filtrado = df_global[df_global['dataset_origin']==origen].copy()
            output_dir_path = os.path.join(LOCAL_DATA_ROOT, nombre_modelo.upper(), f'{origen.upper()}_{str(max_len)}')
            extract_features(df_filtrado, nombre_modelo, tokenizador, modelo, max_len, output_dir_path)

Loading weights: 100%|██████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 5467.31it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extrayendo características con bert (max_len=32) -- MELD: 100%|██████████████████| 13704/13704 [00:49<00:00, 278.05it/s]
Extrayendo características con bert (max_len=32) -- IEMOCAP: 100%|█████████████████| 7515/7515 [00:25<00:00, 293.42it/s]
Extrayendo características con bert (max_len=64) -- MELD: 100%|██████████████████| 13704/13704 [00:49<00:00, 277.98it/s]
Extrayendo características con bert (max_len=64) -- IEMOCAP: 100%|█████████████████| 7515/7515 [00:27<00:00, 276.55it/s]


Loading weights: 100%|██████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 4678.31it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extrayendo características con roberta (max_len=32) -- MELD: 100%|███████████████| 13704/13704 [00:50<00:00, 269.34it/s]
Extrayendo características con roberta (max_len=32) -- IEMOCAP: 100%|██████████████| 7515/7515 [00:26<00:00, 285.05it/s]
Extrayendo características con roberta (max_len=64) -- MELD: 100%|███████████████| 13704/13704 [00:49<00:00, 274.58it/s]
Extrayendo características con roberta (max_len=64) -- IEMOCAP: 100%|██████████████| 7515/7515 [00:27<00:00, 274.72it/s]


Loading weights: 100%|█████████████████████████████████████████████████████████████| 198/198 [00:00<00:00, 23346.89it/s]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading fro

Extrayendo características con deberta (max_len=32) -- MELD: 100%|███████████████| 13704/13704 [01:36<00:00, 142.01it/s]
Extrayendo características con deberta (max_len=32) -- IEMOCAP: 100%|██████████████| 7515/7515 [00:51<00:00, 144.69it/s]
Extrayendo características con deberta (max_len=64) -- MELD: 100%|███████████████| 13704/13704 [01:35<00:00, 143.29it/s]
Extrayendo características con deberta (max_len=64) -- IEMOCAP: 100%|██████████████| 7515/7515 [00:52<00:00, 142.77it/s]


### **Sanity Check**

In [16]:
MODELS = ["BERT", "ROBERTA", "DEBERTA"]
WINDOW_SIZES = [32, 64]
ORIGENES = ['MELD', 'IEMOCAP']

expected_per_origen = {o: len(df_global[df_global['dataset_origin'] == o]) for o in ORIGENES}
expected_total = len(df_global)

print(f"Total esperado: {expected_total} | " + " | ".join([f"{o}: {expected_per_origen[o]}" for o in ORIGENES]))
print()

col_modelo  = 20
col_ventana =  6
col_origen  =  8
col_estado  = 25

header = f"{'MODELO':<{col_modelo}} | {'T':>{col_ventana}} | " + " | ".join([f"{o:>{col_origen}}" for o in ORIGENES]) + f" | {'ESTADO':<{col_estado}}"
print(header)
print("-" * len(header))

all_success = True

for model in MODELS:
    for size in WINDOW_SIZES:
        counts_per_origen = {}
        for origen in ORIGENES:
            folder = os.path.join(LOCAL_DATA_ROOT, model, f"{origen}_{size}")
            if os.path.isdir(folder):
                counts_per_origen[origen] = len([f for f in os.listdir(folder) if f.endswith('.npy')])
            else:
                counts_per_origen[origen] = -1  # Carpeta no existe

        total_found = sum(v for v in counts_per_origen.values() if v >= 0)

        if any(v == -1 for v in counts_per_origen.values()):
            status = "CARPETA MISSING"
            all_success = False
        elif total_found == expected_total and all(counts_per_origen[o] == expected_per_origen[o] for o in ORIGENES):
            status = "OK"
        else:
            diffs = ", ".join([f"{o}:{counts_per_origen[o]}/{expected_per_origen[o]}" for o in ORIGENES if counts_per_origen[o] != expected_per_origen[o]])
            status = f"DIFERENCIA ({diffs})"
            all_success = False

        counts_str = " | ".join([f"{counts_per_origen[o]:>{col_origen}}" if counts_per_origen[o] >= 0 else f"{'MISS':>{col_origen}}" for o in ORIGENES])
        print(f"{model:<{col_modelo}} | {size:>{col_ventana}} | {counts_str} | {status:<{col_estado}}")

        # Ejemplo aleatorio por origen
        for origen in ORIGENES:
            folder = os.path.join(LOCAL_DATA_ROOT, model, f"{origen}_{size}")
            if os.path.isdir(folder):
                npy_files = [f for f in os.listdir(folder) if f.endswith('.npy')]
                if npy_files:
                    sample_name = random.choice(npy_files)
                    data = np.load(os.path.join(folder, sample_name))
                    dim_ok = "Ok" if data.shape == (size, 768) else f"Error (esperado ({size}, 768))"
                    print(f"   Ejemplo {origen} → {sample_name}: shape={data.shape} {dim_ok}")
                    
print("-" * len(header))
if all_success:
    print("\nTodos los embeddings coinciden exactamente con el dataset.")
else:
    print("\nSe encontraron discrepancias.")

Total esperado: 21219 | MELD: 13704 | IEMOCAP: 7515

MODELO               |      T |     MELD |  IEMOCAP | ESTADO                   
-------------------------------------------------------------------------------
BERT                 |     32 |    13704 |     7515 | OK                       
   Ejemplo MELD → MELD_train_dia556_utt4.npy: shape=(32, 768) Ok
   Ejemplo IEMOCAP → IEMOCAP_Ses02M_script03_2_M027.npy: shape=(32, 768) Ok
BERT                 |     64 |    13704 |     7515 | OK                       
   Ejemplo MELD → MELD_test_dia207_utt1.npy: shape=(64, 768) Ok
   Ejemplo IEMOCAP → IEMOCAP_Ses03M_impro05a_M010.npy: shape=(64, 768) Ok
ROBERTA              |     32 |    13704 |     7515 | OK                       
   Ejemplo MELD → MELD_dev_dia97_utt3.npy: shape=(32, 768) Ok
   Ejemplo IEMOCAP → IEMOCAP_Ses03M_script01_3_M046.npy: shape=(32, 768) Ok
ROBERTA              |     64 |    13704 |     7515 | OK                       
   Ejemplo MELD → MELD_train_dia337_utt14.npy: sha